# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
import os
import urllib.request
import time

corpus_dir = "gutenberg_corpus_100"
os.makedirs(corpus_dir, exist_ok=True)

print("Descargando una muestra de 100 libros de forma rápida y directa...")

# Descargamos 100 libros secuenciales usando IDs conocidos para asegurar que existen
count = 0
book_id = 11  # Comenzamos con un ID estándar (ej. Alice in Wonderland)

while count < 100:
    url = f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt"
    file_path = os.path.join(corpus_dir, f"book_{book_id}.txt")

    try:
        # Intentamos descargar el archivo de texto directamente
        urllib.request.urlretrieve(url, file_path)
        count += 1
        if count % 10 == 0:
            print(f"Progreso: {count}/100 libros descargados...")
    except Exception:
        # Si el formato anterior falla, probamos con el formato alternativo sin '-0'
        try:
            alt_url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
            urllib.request.urlretrieve(alt_url, file_path)
            count += 1
            if count % 10 == 0:
                print(f"Progreso: {count}/100 libros descargados...")
        except Exception:
            pass # Si no está disponible, saltamos al siguiente ID

    book_id += 1
    time.sleep(0.1) # Pausa amigable para evitar saturar el servidor

print(f"\n¡Completado! Se descargaron {count} documentos directamente en la carpeta '{corpus_dir}'.")

Descargando una muestra de 100 libros de forma rápida y directa...
Progreso: 10/100 libros descargados...
Progreso: 20/100 libros descargados...
Progreso: 30/100 libros descargados...
Progreso: 40/100 libros descargados...
Progreso: 50/100 libros descargados...
Progreso: 60/100 libros descargados...
Progreso: 70/100 libros descargados...
Progreso: 80/100 libros descargados...
Progreso: 90/100 libros descargados...
Progreso: 100/100 libros descargados...

¡Completado! Se descargaron 100 documentos directamente en la carpeta 'gutenberg_corpus_100'.


In [2]:
import re
import pandas as pd

In [3]:
corpus = []
for filename in sorted(os.listdir(corpus_dir)):
    if filename.endswith(".txt"):
        file_path = os.path.join(corpus_dir, filename)
        doc_id = filename.split(".")[0] # ej: book_11

        try:
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            lines = [line.strip() for line in text.split("\n") if line.strip()]
            title = lines[0] if lines else doc_id
            if len(title) > 50:
                title = title[:47] + "..."

            corpus.append({
                "id": doc_id,
                "title": title,
                "text": text
            })
        except Exception as e:
            print(f"Error leyendo {filename}: {e}")

In [4]:
data = []
doc_lengths = []
all_tokens = set()

In [5]:
for doc in corpus:
    tokens = re.findall(r'\b\w+\b', doc['text'].lower())
    unique_terms = len(set(tokens))
    doc_lengths.append(len(tokens))
    all_tokens.update(tokens)

    data.append({
        "Identificador de documento": doc['id'],
        "Título": doc['title'],
        "Número de términos únicos": unique_terms
    })

In [6]:
df_corpus = pd.DataFrame(data)

In [ ]:
print("Tabla Resumen del Corpus\n")

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(df_corpus.to_markdown(index=False))

print("\n Estadísticas Globales del Corpus \n")
print(f"- Número de documentos: {len(corpus)}")
print(f"- Tamaño del vocabulario: {len(all_tokens)}")
print(f"- Longitud mínima de documento: {min(doc_lengths)} tokens")
print(f"- Longitud media de documento: {sum(doc_lengths) / len(doc_lengths):.2f} tokens")
print(f"- Longitud máxima de documento: {max(doc_lengths)} tokens")

Tabla Resumen del Corpus

| Identificador de documento   | Título                                             |   Número de términos únicos |
|:-----------------------------|:---------------------------------------------------|----------------------------:|
| book_100                     | *** START OF THE PROJECT GUTENBERG EBOOK 100 ***   |                       25752 |
| book_101                     | *** START OF THE PROJECT GUTENBERG EBOOK 101 ***   |                       11337 |
| book_102                     | *** START OF THE PROJECT GUTENBERG EBOOK 102 ***   |                        6430 |
| book_103                     | *** START OF THE PROJECT GUTENBERG EBOOK 103 ***   |                        6854 |
| book_104                     | *** START OF THE PROJECT GUTENBERG EBOOK 104 ***   |                         738 |
| book_105                     | *** START OF THE PROJECT GUTENBERG EBOOK 105 ***   |                        5822 |
| book_106                     | *** START OF 

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [11]:
import math
from collections import Counter

In [12]:
def compute_tf(document_tokens):
    return Counter(document_tokens)

In [13]:
def compute_df(corpus_tokens_list):
    df = Counter()
    for doc_tokens in corpus_tokens_list:
        unique_terms = set(doc_tokens)
        for term in unique_terms:
            df[term] += 1
    return df

In [14]:
def compute_idf(df_dict, num_docs):
    idf = {}
    for term, df_val in df_dict.items():
        if df_val > 0:
            idf[term] = math.log(num_docs / df_val) + 1
        else:
            idf[term] = 0.0
    return idf

In [15]:
def compute_tfidf_weights(tf_dict, idf_dict):
    return {term: tf_val * idf_dict.get(term, 0.0) for term, tf_val in tf_dict.items()}

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

#### Paso 1: Construir un vocabulario con un orden definido

In [25]:
corpus_tokens_list = [re.findall(r'\b\w+\b', doc['text'].lower()) for doc in corpus]
num_docs = len(corpus)

vocabulary = sorted(list(set(term for doc_tokens in corpus_tokens_list for term in doc_tokens)))
vocab_index = {term: idx for idx, term in enumerate(vocabulary)}

print(f"Tamaño del vocabulario: {len(vocabulary)} términos únicos")

Tamaño del vocabulario: 362133 términos únicos


#### Paso 2: Contar las frecuencias TF

In [26]:
tf_dicts = [compute_tf(doc_tokens) for doc_tokens in corpus_tokens_list]
print(f"Frecuencias TF calculadas para {len(tf_dicts)} documentos.")

Frecuencias TF calculadas para 100 documentos.


#### Paso 3: Calcular DF contando cada documento una sola vez por término

In [27]:
df_global = compute_df(corpus_tokens_list)
print(f"Document Frequency (DF) calculada para {len(df_global)} términos.")

Document Frequency (DF) calculada para 362133 términos.


#### Paso 4: Calcular IDF con la función de la sección anterior

In [28]:
idf_global = compute_idf(df_global, num_docs)
print(f"Inverse Document Frequency (IDF) calculada para todo el vocabulario.")

Inverse Document Frequency (IDF) calculada para todo el vocabulario.


#### Paso 5: Obtener la matriz documento-término TF-IDF

In [29]:
import numpy as np

tfidf_matrix_explicit = np.zeros((num_docs, len(vocabulary)))

for i, tf_dict in enumerate(tf_dicts):
    for term, freq in tf_dict.items():
        if term in vocab_index:
            j = vocab_index[term]
            tfidf_matrix_explicit[i, j] = freq * idf_global[term]

print(f"Matriz documento-término TF-IDF construida de forma explícita. Forma: {tfidf_matrix_explicit.shape}")

Matriz documento-término TF-IDF construida de forma explícita. Forma: (100, 362133)


In [30]:
import numpy as np

corpus_tokens_list = [re.findall(r'\b\w+\b', doc['text'].lower()) for doc in corpus]
num_docs = len(corpus)

vocabulary = sorted(list(set(term for doc_tokens in corpus_tokens_list for term in doc_tokens)))
vocab_index = {term: idx for idx, term in enumerate(vocabulary)}

df_global = compute_df(corpus_tokens_list)

idf_global = compute_idf(df_global, num_docs)

tfidf_matrix_explicit = np.zeros((num_docs, len(vocabulary)))

for i, doc_tokens in enumerate(corpus_tokens_list):
    tf_dict = compute_tf(doc_tokens)
    for term, freq in tf_dict.items():
        if term in vocab_index:
            j = vocab_index[term]
            # TF-IDF = TF * IDF
            tfidf_matrix_explicit[i, j] = freq * idf_global[term]

print(f"Matriz explícita construida con forma: {tfidf_matrix_explicit.shape}")

Matriz explícita construida con forma: (100, 362133)


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    analyzer=lambda x: x,
    vocabulary=vocabulary
)

tfidf_matrix_sklearn = vectorizer.fit_transform(corpus_tokens_list).toarray()

max_diff = np.max(np.abs(tfidf_matrix_explicit - tfidf_matrix_sklearn))

print(f"Vocabulario detectado por sklearn: {len(vectorizer.get_feature_names_out())}")
print(f"Vocabulario de nuestra implementación: {len(vocabulary)}")
print(f"Máxima diferencia absoluta entre la matriz explícita y sklearn: {max_diff}")

if max_diff < 1e-5:
    print("¡Verificación exitosa! Las matrices coinciden perfectamente dentro de la tolerancia.")
else:
    print("Atención: Existe una diferencia superior a la esperada entre las matrices.")

Vocabulario detectado por sklearn: 362133
Vocabulario de nuestra implementación: 362133
Máxima diferencia absoluta entre la matriz explícita y sklearn: 0.0
¡Verificación exitosa! Las matrices coinciden perfectamente dentro de la tolerancia.


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [54]:
from sklearn.metrics.pairwise import cosine_similarity

queries = [
    {
        "text": "alice wonderland queen rabbit",
        "intent": "Buscar historias de fantasía clásica, personajes fantásticos y aventuras en mundos irreales."
    },
    {
        "text": "history war battle army",
        "intent": "Encontrar documentos históricos, relatos de conflictos bélicos y campañas militares."
    },
    {
        "text": "love romance heart marriage",
        "intent": "Localizar novelas de drama romántico, relaciones interpersonales y tramas sentimentales."
    },
    {
        "text": "science world earth nature",
        "intent": "Explorar textos científicos, descripciones del entorno natural o ensayos sobre el planeta."
    }
]

vocab_index = {term: idx for idx, term in enumerate(vocabulary)}

In [55]:
from IPython.display import display, Markdown

vectorizer_aligned = TfidfVectorizer(
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    analyzer=lambda x: x,
    vocabulary=vocabulary
)
tfidf_matrix_sklearn_aligned = vectorizer_aligned.fit_transform(corpus_tokens_list).toarray()
for idx_q, q_info in enumerate(queries, start=1):
    q_text = q_info["text"]

    q_tokens = re.findall(r'\b\w+\b', q_text.lower())
    q_tf = compute_tf(q_tokens)

    q_vector = np.zeros(len(vocabulary))
    for term, freq in q_tf.items():
        if term in vocab_index:
            j = vocab_index[term]
            q_vector[j] = freq * idf_global[term]

    q_norm = np.linalg.norm(q_vector)
    doc_norms = np.linalg.norm(tfidf_matrix_explicit, axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        manual_sims = np.dot(tfidf_matrix_explicit, q_vector) / (doc_norms * q_norm)
        manual_sims = np.nan_to_num(manual_sims)

    best_doc_idx = np.argmax(manual_sims)
    best_doc = corpus[best_doc_idx]
    d_vector = tfidf_matrix_explicit[best_doc_idx]

    doc_tokens_set = set(corpus_tokens_list[best_doc_idx])
    q_tokens_set = set(q_tokens)
    shared_terms = sorted(list(q_tokens_set.intersection(doc_tokens_set)))

    partial_products = {}
    for term in shared_terms:
        idx_term = vocab_index[term]
        partial_products[term] = q_vector[idx_term] * d_vector[idx_term]

    dot_product = np.dot(d_vector, q_vector)
    doc_norm = doc_norms[best_doc_idx]
    similitud = manual_sims[best_doc_idx]

    shared_str = ", ".join(shared_terms) if shared_terms else "Ninguno"
    products_str = ", ".join([f"**{k}**: {v:.4f}" for k, v in partial_products.items()]) if partial_products else "-"
    norm_str = f"Doc: {doc_norm:.4f} <br> Q: {q_norm:.4f}"

    markdown_output = f"""
### Consulta {idx_q}: "{q_text}"
* **Intención de búsqueda**: {q_info['intent']}
* **Documento más relevante**: `{best_doc['id']}` - *{best_doc['title']}*

| Términos compartidos | Productos parciales | Producto punto | Normas | Similitud |
| :--- | :--- | :---: | :---: | :---: |
| {shared_str} | {products_str} | {dot_product:.4f} | {norm_str} | **{similitud:.4f}** |

---
"""
    display(Markdown(markdown_output))


### Consulta 1: "alice wonderland queen rabbit"
* **Intención de búsqueda**: Buscar historias de fantasía clásica, personajes fantásticos y aventuras en mundos irreales.
* **Documento más relevante**: `book_12` - **** START OF THE PROJECT GUTENBERG EBOOK 12 ****

| Términos compartidos | Productos parciales | Producto punto | Normas | Similitud |
| :--- | :--- | :---: | :---: | :---: |
| alice, queen, wonderland | **alice**: 3186.6898, **queen**: 482.5327, **wonderland**: 11.6141 | 3680.8366 | Doc: 3288.9438 <br> Q: 5.1293 | **0.2182** |

---



### Consulta 2: "history war battle army"
* **Intención de búsqueda**: Encontrar documentos históricos, relatos de conflictos bélicos y campañas militares.
* **Documento más relevante**: `book_73` - **** START OF THE PROJECT GUTENBERG EBOOK 73 ****

| Términos compartidos | Productos parciales | Producto punto | Normas | Similitud |
| :--- | :--- | :---: | :---: | :---: |
| army, battle, history, war | **army**: 88.2887, **battle**: 125.5256, **history**: 5.5217, **war**: 62.3462 | 281.6823 | Doc: 5047.9336 <br> Q: 2.7833 | **0.0200** |

---



### Consulta 3: "love romance heart marriage"
* **Intención de búsqueda**: Localizar novelas de drama romántico, relaciones interpersonales y tramas sentimentales.
* **Documento más relevante**: `book_109` - **** START OF THE PROJECT GUTENBERG EBOOK 109 ****

| Términos compartidos | Productos parciales | Producto punto | Normas | Similitud |
| :--- | :--- | :---: | :---: | :---: |
| heart, love | **heart**: 21.5443, **love**: 29.0132 | 50.5575 | Doc: 711.4575 <br> Q: 3.1556 | **0.0225** |

---



### Consulta 4: "science world earth nature"
* **Intención de búsqueda**: Explorar textos científicos, descripciones del entorno natural o ensayos sobre el planeta.
* **Documento más relevante**: `book_26` - **** START OF THE PROJECT GUTENBERG EBOOK 26 ****

| Términos compartidos | Productos parciales | Producto punto | Normas | Similitud |
| :--- | :--- | :---: | :---: | :---: |
| earth, nature, science, world | **earth**: 293.2647, **nature**: 89.0496, **science**: 2.8001, **world**: 112.5557 | 497.6701 | Doc: 6949.2365 <br> Q: 2.5787 | **0.0278** |

---


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [57]:
from IPython.display import display, Markdown

def get_snippet(text, query_tokens, length=120):
    """Extrae un fragmento del texto que contiene alguno de los tokens de la consulta."""
    text_lower = text.lower()
    for token in query_tokens:
        idx = text_lower.find(token)
        if idx != -1:
            start = max(0, idx - 40)
            end = min(len(text), idx + length)
            snippet = text[start:end].replace("\n", " ").strip()
            return f"...{snippet}..."
    return text[:length].replace("\n", " ").strip() + "..."

for idx_q, q_info in enumerate(queries, start=1):
    q_text = q_info["text"]
    q_tokens = re.findall(r'\b\w+\b', q_text.lower())

    q_vector = np.zeros(len(vocabulary))
    q_tf = compute_tf(q_tokens)
    for term, freq in q_tf.items():
        if term in vocab_index:
            j = vocab_index[term]
            q_vector[j] = freq * idf_global[term]

    q_norm = np.linalg.norm(q_vector)
    doc_norms = np.linalg.norm(tfidf_matrix_explicit, axis=1)

    with np.errstate(divide='ignore', invalid='ignore'):
        manual_sims = np.dot(tfidf_matrix_explicit, q_vector) / (doc_norms * q_norm)
        manual_sims = np.nan_to_num(manual_sims)

    sklearn_sims = cosine_similarity([q_vector], tfidf_matrix_sklearn_aligned)[0]

    ranked_indices = np.argsort(manual_sims)[::-1]

    table_rows = []
    for rank_pos, doc_idx in enumerate(ranked_indices[:5], start=1):
        doc_obj = corpus[doc_idx]
        score_manual = manual_sims[doc_idx]
        score_sklearn = sklearn_sims[doc_idx]

        assert abs(score_manual - score_sklearn) < 1e-5, f"Discrepancia de tolerancia en doc {doc_obj['id']}"

        snippet = get_snippet(doc_obj["text"], q_tokens)
        title_clean = doc_obj["title"].replace("|", "-").strip()

        table_rows.append(f"| {rank_pos} | {doc_obj['id']} | {title_clean[:50]} | {score_manual:.4f} | {snippet} |")

    markdown_table = f"""
### Top-5 Ranking para Consulta {idx_q}: "{q_text}"
* **Intención de búsqueda**: {q_info['intent']}

| Posición | Identificador | Título | Puntuación | Fragmento |
| :---: | :--- | :--- | :---: | :--- |
""" + "\n".join(table_rows) + "\n\n---\n"

    display(Markdown(markdown_table))


### Top-5 Ranking para Consulta 1: "alice wonderland queen rabbit"
* **Intención de búsqueda**: Buscar historias de fantasía clásica, personajes fantásticos y aventuras en mundos irreales.

| Posición | Identificador | Título | Puntuación | Fragmento |
| :---: | :--- | :--- | :---: | :--- |
| 1 | book_12 | *** START OF THE PROJECT GUTENBERG EBOOK 12 *** | 0.2182 | ...THROUGH THE LOOKING-GLASS  And What Alice Found There  By Lewis Carroll  The Millennium Fulcrum Edition 1.7     DRAMATIS PERSONÆ. (_As arranged before comme... |
| 2 | book_11 | *** START OF THE PROJECT GUTENBERG EBOOK 11 *** | 0.2069 | ...ENBERG EBOOK 11 ***  [Illustration]     Alice’s Adventures in Wonderland  by Lewis Carroll  THE MILLENNIUM FULCRUM EDITION 3.0  Contents   CHAPTER I.     Down t... |
| 3 | book_77 | *** START OF THE PROJECT GUTENBERG EBOOK 77 *** | 0.0100 | ...WINDOW  XII. THE DAGUERREOTYPIST  XIII. ALICE PYNCHEON  XIV. PHŒBE’S GOOD-BYE  XV. THE SCOWL AND SMILE  XVI. CLIFFORD’S CHAMBER  XVII. THE FLIGHT OF TWO OWLS  X... |
| 4 | book_100 | *** START OF THE PROJECT GUTENBERG EBOOK 100 *** | 0.0072 | ...mine own direct knowledge, without any malice, but to speak of him as my kinsman, he’s a most notable coward, an infinite and endless liar, an hourly promise-br... |
| 5 | book_78 | *** START OF THE PROJECT GUTENBERG EBOOK 78 *** | 0.0054 | ...r hand, he had been married to the Hon. Alice Rutherford for scarce a three months, and it was the thought of taking this fair young girl into the dangers and i... |

---



### Top-5 Ranking para Consulta 2: "history war battle army"
* **Intención de búsqueda**: Encontrar documentos históricos, relatos de conflictos bélicos y campañas militares.

| Posición | Identificador | Título | Puntuación | Fragmento |
| :---: | :--- | :--- | :---: | :--- |
| 1 | book_73 | *** START OF THE PROJECT GUTENBERG EBOOK 73 *** | 0.0200 | ...les. There was a portion of the world’s history which he had regarded as the time of wars, but it, he thought, had been long gone over the horizon and had disap... |
| 2 | book_66 | The Project Gutenberg eBook of The Dawn of Amat... | 0.0153 | ...he demonstrated, for the first time in history, the principle of electromagnetic induction.  He had constructed powerful electromagnets consisting of coils of... |
| 3 | book_67 | The Project Gutenberg eBook of The Black Experi... | 0.0149 | ...st several years, the study of American history has turned a new direction.  Previously, it emphasized how the various immigrant groups in America shed their di... |
| 4 | book_104 | *** START OF THE PROJECT GUTENBERG EBOOK 104 *** | 0.0121 | ...sk as we would treat the emergency of a war, but at the same time, through this employment, accomplishing greatly needed projects to stimulate and reorganize th... |
| 5 | book_54 | ﻿The Project Gutenberg eBook of The Marvellous ... | 0.0120 | .... Woggle-Bug, T. E.  A Highly Magnified History  Old Mombi indulges in Witchcraft  The Prisoners of the Queen  The Scarecrow Takes Time to Think  The Astonishin... |

---



### Top-5 Ranking para Consulta 3: "love romance heart marriage"
* **Intención de búsqueda**: Localizar novelas de drama romántico, relaciones interpersonales y tramas sentimentales.

| Posición | Identificador | Título | Puntuación | Fragmento |
| :---: | :--- | :--- | :---: | :--- |
| 1 | book_109 | *** START OF THE PROJECT GUTENBERG EBOOK 109 *** | 0.0225 | ...a little tavern    Ashes of Life        Love has gone and left me and the days are all alike;    The Little Ghost        I knew her for a little ghost    Kin to... |
| 2 | book_100 | *** START OF THE PROJECT GUTENBERG EBOOK 100 *** | 0.0225 | ...CAESAR     THE TRAGEDY OF KING LEAR     LOVE’S LABOUR’S LOST     THE TRAGEDY OF MACBETH     MEASURE FOR MEASURE     THE MERCHANT OF VENICE     THE MERRY WIVES O... |
| 3 | book_51 | *** START OF THE PROJECT GUTENBERG EBOOK 51 *** | 0.0126 | ...o those that seek them issue forth, For Love in sequel works with Fate, And draws the veil from hidden worth.                     —TENNYSON     to all the girls... |
| 4 | book_32 | *** START OF THE PROJECT GUTENBERG EBOOK 32 *** | 0.0119 | ...-eyes of the cities and parks; a lot of lovely views of streets, of buildings, outside and in, and some of those gorgeous gardens, and, most important of all, o... |
| 5 | book_26 | *** START OF THE PROJECT GUTENBERG EBOOK 26 *** | 0.0115 | ...ecute their airy purposes, And works of love or enmity fulfil. For those the race of Israel oft forsook Their Living Strength, and unfrequented left His righteo... |

---



### Top-5 Ranking para Consulta 4: "science world earth nature"
* **Intención de búsqueda**: Explorar textos científicos, descripciones del entorno natural o ensayos sobre el planeta.

| Posición | Identificador | Título | Puntuación | Fragmento |
| :---: | :--- | :--- | :---: | :--- |
| 1 | book_26 | *** START OF THE PROJECT GUTENBERG EBOOK 26 *** | 0.0278 | ...ce within them as a guide, My umpire Conscience; whom if they will hear, Light after light, well us’d, they shall attain, And to the end, persisting, safe arriv... |
| 2 | book_56 | The Project Gutenberg eBook of NREN for All: In... | 0.0235 | ...y Act charges the White House Office of Science and Technology Policy (OSTP) with coordinating efforts to develop applications for high-performance computing ne... |
| 3 | book_20 | *** START OF THE PROJECT GUTENBERG EBOOK 20 *** | 0.0221 | ...ce within them as a guide My Umpire _Conscience_, whom if they will hear, Light after light well us’d they shall attain, And to the end persisting, safe arrive.... |
| 4 | book_58 | ﻿*** START OF THE PROJECT GUTENBERG EBOOK PARAD... | 0.0214 | ...st made him such? Let his tormentor, Conscience, find him out;     130 For him I was not sent, nor yet to free That people, victor once, now vile and base, Dese... |
| 5 | book_109 | *** START OF THE PROJECT GUTENBERG EBOOK 109 *** | 0.0213 | ...om out.--Ah, fearful pawn!   For my omniscience paid I toll   In infinite remorse of soul.   All sin was of my sinning, all   Atoning mine, and mine the gall... |

---
